In [8]:
import torch
torch.cuda.empty_cache()

In [9]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import requests, time, threading, uvicorn, nest_asyncio
# from deploy_baseline_runpod import app

In [10]:
# nest_asyncio.apply() # Allow nested event loops for Jupyter Notebook
# # Start the FastAPI server in a separate thread
# threading.Thread(
#     target=lambda: uvicorn.run(app, host="0.0.0.0", 
#                                port=8000, log_level="error"), daemon=True).start()
# time.sleep(10) 

In [ ]:
BATCH_SIZES = [4, 16, 32, 64]
results = []

# Replace with yours
URL = "https://9eez2sw4h5b6qd-7860.proxy.runpod.net/benchmarks"

for bs in BATCH_SIZES:
    print(f"--- Stress Testing Batch Size: {bs} ---")
    payload = {
        "input_tokens": 512,
        "generated_tokens": 128,
        "batch_size": bs,
        "num_prompts": 125
    }
    try:
        r = requests.post(URL, json=payload, timeout=400).json()

        results.append(
            {
                "batch_size": bs,
                "Global_TPS": r["global_tps"],
                # TTFT: Time To First Token
                "TTFT_Mean": r["ttft"]["mean"], 
                "TTFT_Median": r["ttft"]["median"], 
                "TTFT_P95": r["ttft"]["p95"],
                "TTFT_P99": r["ttft"]["p99"],
                # TPOT: Time Per Output Token
                "TPOT_Mean": r['tpot']['mean'],
                "TPOT_Median": r['tpot']['median'], 
                "TPOT_P95": r['tpot']['p95'],
                "TPOT_P99": r['tpot']['p99'],
                # ITL: Inference Time Latency
                "ITL_Mean": r['itl']['mean'],
                "ITL_Median": r['itl']['median'], 
                "ITL_P95": r['itl']['p95'],
                "ITL_P99": r['itl']['p99'],
                # E2E: End-to-End Latency
                "E2E_Mean": r['e2e']['mean'], 
                "E2E_Median": r['e2e']['median'],
                "E2E_P95": r['e2e']['p95'],
                "E2E_P99": r['e2e']['p99']
            }
        )
    except Exception as e:
        print(f"Error for batch size {bs}: {e}")

--- Stress Testing Batch Size: 4 ---
--- Stress Testing Batch Size: 16 ---
--- Stress Testing Batch Size: 32 ---
--- Stress Testing Batch Size: 64 ---


In [ ]:
results

In [ ]:
df = pd.DataFrame(results)

In [ ]:
df.set_index('batch_size').T

In [ ]:
df.to_csv(f"results/phi4_final_bench_{time.strftime('%Y%m%d')}.csv", index=False)

In [ ]:
# Plots
sns.set_theme(style="whitegrid")

fig,ax = plt.subplots(2, 3, figsize=(20, 10))

# Global TPS
sns.lineplot(x="batch_size", y="Global_TPS", data=df, marker="o", ax=ax[0,0])
ax[0,0].set_title("Global TPS vs Batch Size")

# TTFT Latency
sns.lineplot(x="batch_size", y="TTFT_Mean", data=df, marker="o", label="TTFT Mean", ax=ax[0,1])
sns.lineplot(x="batch_size", y="TTFT_Median", data=df, marker="x", label="TTFT Median", ax=ax[0,1])
sns.lineplot(x="batch_size", y="TTFT_P95", data=df, marker="*", label="TTFT P95", ax=ax[0,1])
sns.lineplot(x="batch_size", y="TTFT_P99", data=df, marker="*", label="TTFT P99", ax=ax[0,1])
ax[0,1].set_title("TTFT vs Batch Size")
ax[0,1].legend()

# TPOT Latency
sns.lineplot(x="batch_size", y="TPOT_Mean", data=df, marker="o", label="TPOT Mean", ax=ax[0,2])
sns.lineplot(x="batch_size", y="TPOT_Median", data=df, marker="x", label="TPOT Median", ax=ax[0,2])
sns.lineplot(x="batch_size", y="TPOT_P95", data=df, marker="*", label="TPOT P95", ax=ax[0,2])
sns.lineplot(x="batch_size", y="TPOT_P99", data=df, marker="*", label="TPOT P99", ax=ax[0,2])
ax[0,2].set_title("TPOT vs Batch Size")
ax[0,2].legend()

# ITL Latency
sns.lineplot(x="batch_size", y="ITL_Mean", data=df, marker="o", label="ITL Mean", ax=ax[1,0])
sns.lineplot(x="batch_size", y="ITL_Median", data=df, marker="x", label="ITL Median", ax=ax[1,0])
sns.lineplot(x="batch_size", y="ITL_P95", data=df, marker="*", label="ITL P95", ax=ax[1,0])
sns.lineplot(x="batch_size", y="ITL_P99", data=df, marker="*", label="ITL P99", ax=ax[1,0])
ax[1,0].set_title("ITL vs Batch Size")
ax[1,0].legend()

# E2E Latency
sns.lineplot(x="batch_size", y="E2E_Mean", data=df, marker="o", label="E2E Mean", ax=ax[1,1])
sns.lineplot(x="batch_size", y="E2E_Median", data=df, marker="x", label="E2E Median", ax=ax[1,1])
sns.lineplot(x="batch_size", y="E2E_P95", data=df, marker="*", label="E2E P95", ax=ax[1,1])
sns.lineplot(x="batch_size", y="E2E_P99", data=df, marker="*", label="E2E P99", ax=ax[1,1])
ax[1,1].set_title("E2E vs Batch Size")
ax[1,1].legend()


plt.suptitle("Phi-4 African History Naive Benchmark: Latency Distribution", fontsize=16)
plt.savefig('results/fastapi_benchmark_plots.png')
plt.show()

In [ ]:
# CSV to JSON Conversion (In case json outputs are not shown when saving this file from runpod)
# https://jsonlint.com/csv-to-json